In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import seaborn as sns  # noqa: E402

from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    MusicTypeVariants,
    ExclusionCategories,
)
from src.analysis.mean_variance import (  # noqa: E402
    compute_intersubject_stats,
    compute_windowed_stats,
)
from src.visualization.mean_variance_plots import (  # noqa: E402
    plot_timeseries,
    plot_variance_distribution,
    plot_windowed_analysis,
)
from scripts.analysis_common import load_analyzers, analyzers_to_datasets  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
print("Setup complete.")

# EEG Intersubject Mean-Variance Synchrony Analysis — Broadband

This notebook implements the **broadband (z-scored raw signal)** intersubject mean-variance synchrony pipeline:

* Time-series overview of intersubject mean and variance
* Intersubject variance distribution
* Windowed synchrony detection

All computation uses `src.analysis.mean_variance` and all visualisation uses
`src.visualization.mean_variance_plots`.

> **Parameters to tweak:** `CONDITION`, `MUSIC_TYPES`, `WINDOW_SEC`, `SYNC_PERCENTILE` in the
> *Configuration* cell below.

## Configuration

In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Windowed synchrony parameters ────────────────────────────────────────────
WINDOW_SEC = 2.0  # window length in seconds
STEP_SEC = WINDOW_SEC / 2  # 50% overlap between successive windows
SYNC_PERCENTILE = 10.0  # windows with variance < this percentile are "sync candidates"

# ── Data processing flag ──────────────────────────────────────────────────────
# Set True to load raw EDF files, resample, stack, and save before analysis.
# Keep False to use already-saved concatenated arrays.
process_and_save_data = False

# ── Plot saving ──────────────────────────────────────────────────────────────
# Plots are saved into a 'plots/' subdirectory next to this notebook.
# Set SAVE_PLOTS=False to only display figures inline without saving.
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR / "01-raw-mean-variance-analysis" / "plots" / "broadband"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOTS_DIR}")

## Data Loading

In [ ]:
# Load (or process-and-save) one EEGSummarizedAnalyzer per music type,
# normalise the data to z-scores (axis=2), and convert to AnalysisData.
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    process_and_save_data,
    normalize_data=True,
)
datasets = analyzers_to_datasets(analyzers)
print("Available datasets:", list(datasets.keys()))

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use `ad` and the
derived variables `n_subjects`, `n_channels`, `n_times`.

In [ ]:
LABEL = MusicTypeVariants.CLASSICAL.value
# LABEL = MusicTypeVariants.PSYTRANCE.value

ad = datasets[LABEL]
n_subjects, n_channels, n_times = ad.data.shape
print(f"Dataset : {LABEL}")
print(f"Shape   : {ad.data.shape}  (subjects × channels × time points)")
print(f"Duration: {n_times / ad.sfreq:.1f} s  @  {ad.sfreq} Hz")

---
## Part 1 — Broadband (z-scored raw signal)

For the full broadband signal we compute:
* **Intersubject variance** at each `(channel, time)` cell — low values indicate synchrony
* **Per-channel-averaged time series** of mean and variance
* **Windowed statistics** with synchrony candidate labelling

### 1.1  Compute intersubject statistics

`compute_intersubject_stats` returns a dict with `inter_var`, `inter_mean`, `mean_t`,
`var_t`, `std_t`, and `mean_over_ch`.

In [ ]:
stats = compute_intersubject_stats(ad.data)

print(f"inter_var shape : {stats['inter_var'].shape}  (channels × time)")
print(f"mean_t    shape : {stats['mean_t'].shape}   (time)")
print(f"var_t     mean  : {stats['var_t'].mean():.4f}")
print(f"std_t     mean  : {stats['std_t'].mean():.4f}")

### 1.2  Time-series overview

Two-panel figure:
* **Top** — per-subject channel-average traces (light blue) + group mean (black) with ±1 SD band
* **Bottom** — channel-averaged intersubject variance over time

In [ ]:
fig_ts = plot_timeseries(
    stats,
    ad.sfreq,
    label=LABEL,
    save_path=PLOTS_DIR / "timeseries.png" if SAVE_PLOTS else None,
)

### 1.3  Intersubject variance distribution

Histogram of all `(channel × time)` variance values, clipped at the 99th percentile.

In [ ]:
fig_dist = plot_variance_distribution(
    stats["inter_var"],
    label=LABEL,
    save_path=PLOTS_DIR / "variance_distribution.png" if SAVE_PLOTS else None,
)

### 1.4  Windowed synchrony analysis

The recording is split into overlapping windows of `WINDOW_SEC` seconds
(step = `STEP_SEC`, i.e. 50 % overlap).
A window is flagged as a **synchrony candidate** when its mean intersubject variance
falls below the `SYNC_PERCENTILE`-th percentile of the full `var_t` distribution.

In [ ]:
df_windows = compute_windowed_stats(
    stats,
    n_times=n_times,
    sfreq=ad.sfreq,
    window_sec=WINDOW_SEC,
    sync_percentile=SYNC_PERCENTILE,
    step_sec=STEP_SEC,
)
n_sync = df_windows["sync_candidate"].sum()
print(f"Total windows   : {len(df_windows)}")
print(f"Sync candidates : {n_sync}  ({100 * n_sync / len(df_windows):.1f} %)")
df_windows.head(10)

#### Bar charts — per-window mean signal and mean variance

Green bars = synchrony candidates (mean variance below threshold).

In [ ]:
fig_bar, fig_overlay = plot_windowed_analysis(
    stats,
    df_windows,
    ad.sfreq,
    label=LABEL,
    window_sec=WINDOW_SEC,
    sync_percentile=SYNC_PERCENTILE,
    step_sec=STEP_SEC,
    save_path_bar=PLOTS_DIR / "windowed_bar.png" if SAVE_PLOTS else None,
    save_path_overlay=PLOTS_DIR / "windowed_overlay.png" if SAVE_PLOTS else None,
)